# Constrained Dynamics and Maximal Coordinates

Dynamics computation for articulated bodies generalizes the Newton-Euler equation for a single rigid body. Indeed, one can find the similar linearity with respect to acceleration and parameter dependencies on position and velocity, with the generalized coordinate representation:

$$
\mathbf{\tau} = \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) 
$$

Efficient algorithms like RNEA and ABA can compute $\mathbf{\tau}$ and $\ddot{\mathbf{q}}$ with the same complexity as processing each link one-by-one. The efficiency stems from the recursive structure for which there must "root" and "leaf" links to start the recursion. Basically, the above algorithms work for an _open kinematic structure_ so there should be no loop for traversing from one end to the other. A robot arm apprently fulfils this requirement (left in [](#robokin)). The other example could be a tree-like kinematic chain, for which the backward recursion just needs to sum up the effects from the successor links. Humanoid robots and multi-fingered hands are with such kinematic structures (middle in [](#robokin)). 

```{figure} ../images/robokin.png
---
name: robokin
---

Kinematic structure for modelling various robots: Left - Open chain for an robot arm; Middle - Open tree with the pelvis link as the root for a humanoid robot; Right - Closed structure due to parallel mechanism that cannot be modeled as open kinematic chain/tree. Breaking a joint in the loop, e.g. at the arrow position,  can recover an open structure but the original coupling must be accounted with explicit constraints.
```

However, when there exist closed loops, the kinematic model cannot be handled by straightly applying RNEA and ABA. One way to convert the closed structure back to tractable open chains is to break a joint connecting two links and then explicitly impose a constraint to couple the original joint points that should align. For the instance of the right in [](#robokin), the joint frames on two connecting links should always be aligned except about the revoluting direction. Since the poses of all frames can be obtained from forward kinematics, the constraints reflecting the above coupling can be abstracted as $h(\mathbf{q}) = \mathbf{0}$, and the dynamics computation now needs to jointly solve:

$$
\begin{aligned}
&   \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) = \mathbf{\tau} \\
&   h(\mathbf{q}) = \mathbf{0}
\end{aligned}
$$

This models the motion of rigid bodies under constraints besides the articulated joint constraints. This extends basic dynamic models to _constrained dynamics_. For those who are familiar with Lagrangians of constrained optimization or mechanics, the motion constraints exert extra constraint forces to correct violation of constraints. To see this, let's first differentiate $h(\cdot)$ to make it explicit with $\ddot{\mathbf{q}}$:

$$
\begin{aligned}
&   h({\mathbf{q}}) = \mathbf{0}  \Rightarrow \mathbf{J}_h(\mathbf{q})\dot{\mathbf{q}} = \mathbf{0}   \\
\Rightarrow &   \mathbf{J}_h(\mathbf{q})\ddot{\mathbf{q}} + \dot{\mathbf{J}_h}(\mathbf{q})\dot{\mathbf{q}} = \mathbf{0}
\end{aligned}
$$
where $\mathbf{J}_h$ is the Jacobian of $h(\cdot)$ (Do not confuse it with the Jacobian of forward kinematics). The acceleration of the constrained system can be seen as the closest one to the unconstrained system while subject to the acceleration constraints:

$$
\begin{aligned}
&   \arg\min\limits_{\ddot{\mathbf{q}}} \|   \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) - \mathbf{\tau}  \|_{\mathbf{M}^{-1}}    \\
\text{s.t.} \quad&   \mathbf{J}_h(\mathbf{q})\ddot{\mathbf{q}} + \dot{\mathbf{J}_h}(\mathbf{q})\dot{\mathbf{q}} = \mathbf{0}
\end{aligned}
$$
with $\| \mathbf{x} \|_{\mathbf{Q}} = \mathbf{x}^{\top}\mathbf{Q}\mathbf{x}$. Constructing the Lagrangian and deriving the first-order optimality condition:

$$
\begin{aligned}
&   \mathcal{L}(\mathbf{q}, \mathbf{\lambda}) =  \|   \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) - \mathbf{\tau}  \|_{\mathbf{M}^{-1}} + \mathbf{\lambda}^{\top} [\mathbf{J}_h(\mathbf{q})\ddot{\mathbf{q}} + \dot{\mathbf{J}_h}(\mathbf{q})\dot{\mathbf{q}}]   \\
\Rightarrow &   \frac{\partial \mathcal{L}}{\partial \ddot{\mathbf{q}}} = \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) - \mathbf{\tau} + \mathbf{J}_h^{\top}(\mathbf{q})\mathbf{\lambda} = \mathbf{0}    \\
\Rightarrow &   \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) = \mathbf{\tau} - \mathbf{J}_h^{\top}(\mathbf{q})\mathbf{\lambda}
\end{aligned}
$$
One can find that the Lagrangian multiplier $\mathbf{\lambda}$ is the extra force due to the constraint while $\mathbf{J}_h(\mathbf{q})$ converts it to the generalized coordinate space. It is possible to assemble the above equation with the constraint, yielding a more compact form:

$$
\begin{bmatrix}
\mathbf{M}(\mathbf{q})  &   \mathbf{J}_h^{\top}(\mathbf{q}) \\
\mathbf{J}_h(\mathbf{q})    &   \mathbf{0}
\end{bmatrix}
\begin{bmatrix}
\ddot{\mathbf{q}}   \\
\mathbf{\lambda}    
\end{bmatrix}   +   
\begin{bmatrix}
\mathbf{C}(\dot{\mathbf{q}}, \mathbf{q})    \\
\dot{\mathbf{J}_h}(\mathbf{q})\dot{\mathbf{q}}
\end{bmatrix}   =
\begin{bmatrix}
\mathbf{I}  \\
\mathbf{0}
\end{bmatrix}\tau
$$
Again, the equation looks similar to the Newton-Euler form with the mass matrix now assembles constraint Jacobian and the bias term augments the part similar to the Colioris and centrifugal forces. For forward dynamics, the unknowns are still with linear equations.

Constrained dynamics extend articulated body dynamics from describing free motion to involving interactions external to the articulated bodies. They are not only useful for coping with parallel mechanisms but actually ubiquitous in robot simulation. When links are in contact with others in the environment, their free motions are naturally constrained. Unlike constraints due to articulation, contact constraints are not necessarily bilateral to admit equality constraints like $h(\mathbf{q}) = 0$, and the constraint forces due to contacts are usually subject to friction limits. Resolving [](#eq-constrsys) needs to impose constraints on $\mathbf{\lambda}$. Therefore modelling contacts and collisions are central to build fast and realistic simulation, and have been a long-standing research question in the domain of graphics and robotics. For a more thorough tutorial on this topic, it is highly recommended to refer to [@SiggraphContact22]. 

The fact that one can use explicit constraints to account for articulation effects raises a question: can we replace the generalized coordinate with all the link poses and velocity subject to articulation constraints as the state? The answer is a big yes and doing so can be beneficial in certain scenarios. For instance, when most objects in the scene are separated blocks and very few of them are within an articulated chain, it would make more sense to directly apply the original form of Newton-Euler equation to block pose and velocity. Such a representation is called __maximal coordinate__ in contrast to the generalized coordinate in the joint space, and has main usage in animation and game engine. Robotics usually stick to generalized coordinates because the scene is usually less dominated by many free objects, the motion is inherently conforming to articulation constraints, and the embedded sensors often measure joint angles between links instead of the spatial state. However, research has found that maximal coordinate can sometimes benefit robot learning as a representation more amenable to machine learning or optimization algorithms [@explicitconstr_nips2020] [@maximallqr_icra2021]. 

# Software Practice and Model-based Analysis
Simulating rigid body dynamics is fundamental to robotics and animation hence numerous software implementations are out there. Prominent open-source libraries include [Open Dynamics Engine](https://www.ode.org), [Bullet](https://github.com/bulletphysics/bullet3), [Mujoco](https://github.com/google-deepmind/mujoco), [DART](https://dartsim.github.io), [Pinocchio](https://github.com/stack-of-tasks/pinocchio), [Drake](https://drake.mit.edu) and many more. Proprietary software like [Nvidia PhysX/Isaac Sim](https://developer.nvidia.com/physx-sdk) are also commonly used. They all implement dynamic algorithms covered above, as well as various contact models, solvers, integrators and interface to model files and visualization. Some of them like Bullet, Mujoco and Nvidia Isaac Sim have already been extensively used as the backbone of reinforcement learning environments. 

The flourish of simulation software greatly help researchers and practitioners to quickly prototype models through API lists. However, grasping how the simulation at hand is built and what numerics are undergoing can still be vital for efficiently creating, tuning and debugging desired behaviours. For instance, inversing mass matrices that are poorly conditioned could cause instability. Users might not be aware that their mechanisms are forming matrices like these and find models exploding under default simulaiton parameters. Sometimes it is necessary to make trade-off between stable simulation and numeric precision. Knowing which aspects will be compromised due to e.g. modulating a constraint solver can be critical knowhow for not sacrificing too much fidelity while stabilizing the simulation. One may find information about these from simulators' documentation but probably not always something in-depth. For more thorough understanding, it may be worth to look at relevant literature about the numeric origin of simulation, e.g. for Mujoco [@todorov2012mujoco] and Nvidia Isaac Sim [@macklinthesis].

Treating simulators as a white-box is also necessary for analysis before running a model instance. The equation form like [](#eq-constrsys) underpins the math structure of a wide range of dynamic simulation of rigid bodies, no matter how many links are involved and how complicated they are interacting with each other. The linearity with $\ddot{\mathbf{q}}$ and $\mathbf{\tau}$ opens the possibility of leveraging linear algebra tools to analyze how the system will evolve in the longrun before initiating the actual simulation. 

Conclusions can be made about $\mathbf{M}$ and the Coriolis and centrifugal term $\mathbf{C}$ because how they are computed for rigid body systems. It is obvious that $\mathbf{M}$ is symmetric and positive-semidefinite while it is probably less apparent to see that $\dot{\mathbf{M}} - 2\mathbf{C}$ is skew-symmetric. The latter and its proof can be found as proposition 8.1 in [@modernrobotics]. This states about the _passivity_ of rigid body systems, namely the system as a whole will not yield energy more than the input due to $\mathbf{\tau}$. Passivity is a critical component to prove that we can design some $\mathbf{\tau}$ to eventually bring $\mathbf{q}$ to the desired configuration, even when simulation parameters are different from the real system. 

One can also tell from the system equation like [](#eq-constrsys) about the difficulty of finding $\mathbf{\tau}$ per task requirements. For instance, the identity matrix $\mathbf{I}$ multiplying $\mathbf{\tau}$ implies one can directly apply forces to all system coordinates. A more general form will be:

$$
\mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) = \mathbf{B}\mathbf{\tau}
$$
for which $\mathbf{B}$ is not necessarily full-rank as $\mathbf{I}$ and may have some zero rows. This models the situation when some coordinates are not directly actuated by $\mathbf{\tau}$ but might be indirectly influenced by the interaction with other coordinate dimensions. A concrete example is the CartPole environment in [the previous module](../1_RLSystem/notebooks.ipynb), for which no motor is attached to the revolute joint and the pole link must be manipulated via sliding the cart link. This category of systems are called __underactuated system__. Underactuated systems are challenging to control, even though replacing $\mathbf{I}$ to $\mathbf{B}$ changes little to forward dynamics simulation. Many interesting robot tasks are underactuated in essence, such as legged locomotion and non-prehensile manipulation. The latter, e.g. an arm pushing an object, is especially representative because $\mathbf{q}$ now comprises of both robot and object links i.e. $\mathbf{q}=[\mathbf{q}_r, \mathbf{q}_o]^\top$ and $\mathbf{q}_o$ can only be driven via constraints established from proper contacts. As a result, tasks entailing rich, multi-step and delicate contacts are still challenging state-of-the-art research. Refer to [revelant materials](https://underactuated.mit.edu) for more thorough treatments on the topic of underactuated robotics. 



```{bibliography}
:filter: docname in docnames
```